# 5. Dimensionality reduction and QC overlays

After batch correction and feature selection, we embed the integrated dataset into low-dimensional spaces for visualization and downstream clustering. Dimensionality reduction helps to summarize major sources of variation across cells and to visually assess sample integration and quality-control patterns.

In this notebook, we:

- perform PCA using deviance-selected genes,
- run t‑SNE and UMAP on the PCA representation, and
- overlay key quality-control metrics on the UMAP embedding (library size, mitochondrial fraction, and doublet scores).

The output is an updated AnnData object with PCA, t‑SNE, and UMAP coordinates stored for downstream clustering and reclustering analyses.

## 5.1. Imports and Scanpy settings

In [ ]:
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

# Scanpy verbosity and plotting
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)

sns.set(style="whitegrid")

## 5.2. Load input AnnData

We load the feature-selected, batch-corrected AnnData object that contains:

- normalized expression layers (e.g. `log1p_norm`),
- feature selection annotations (e.g. `highly_deviant`), and
- QC metrics (e.g. `total_counts`, `pct_counts_mt`, `scDblFinder_score`).

In [ ]:
input_file = "feature_selected.h5ad"
print(f"Reading input AnnData: {input_file}")

adata = sc.read(filename=input_file)
adata

## 5.3. Choose expression layer for embedding

For dimensionality reduction, we use the `log1p_norm` layer as the expression representation. We also use the deviance-based feature selection (`highly_deviant`) to define which genes contribute to PCA.

** Here we use the `log1p_norm` layer (derived from scran-normalized counts) as the working expression space for PCA and UMAP.

In [ ]:
# Use the normalized representation for dimensionality reduction
adata.X = adata.layers["log1p_norm"]

# Tell Scanpy which genes to use as "highly variable"
adata.var["highly_variable"] = adata.var["highly_deviant"].astype(bool)
print(f"Number of genes used for PCA: {adata.var['highly_variable'].sum()}")

## 5.4. Principal component analysis (PCA)

We compute PCA on the deviance-selected genes and visualize the first two PCs colored by total counts. This provides an initial overview of major sources of variation and potential residual QC effects.

In [ ]:
print("Running PCA...")
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=True)

sc.pl.pca_scatter(
    adata,
    color="total_counts",
    title="PCA: colored by total counts",
)
plt.show()

## 5.5. t‑SNE embedding

Next, we compute a t‑SNE embedding using the PCA coordinates as input. t‑SNE is useful for visualizing local neighborhoods and fine-grained structure in the data.

In [ ]:
print("Running t-SNE...")
sc.tl.tsne(adata, use_rep="X_pca")

sc.pl.tsne(
    adata,
    color="total_counts",
    title="t-SNE: colored by total counts",
)
plt.show()

## 5.6. UMAP embedding

We then compute a UMAP embedding based on the PCA representation. UMAP is widely used in single-cell analysis for its balance between preserving global and local structure.

In [ ]:
print("Running UMAP...")
sc.pp.neighbors(adata, use_rep="X_pca")
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color="total_counts",
    title="UMAP: colored by total counts",
)
plt.show()

## 5.7. Quality-control overlays on UMAP

To visually inspect whether QC metrics or doublets drive any residual structure, we overlay several metrics on the same UMAP embedding:

- `total_counts`: library size,
- `pct_counts_mt`: mitochondrial fraction,
- `scDblFinder_score` and `scDblFinder_class`: doublet scores and classifications.

In [ ]:
print("Inspecting quality control metrics on UMAP...")
sc.pl.umap(
    adata,
    color=["total_counts", "pct_counts_mt", "scDblFinder_score", "scDblFinder_class"],
    wspace=0.4,
)
plt.show()

## 5.8. Save dimensionality-reduced AnnData and next steps

We now save the AnnData object with PCA, t‑SNE, and UMAP coordinates stored in `.obsm`. This object will be used in the next notebook for clustering (Leiden), reclustering of specific populations (e.g. GABAergic nuclei), and further downstream analyses.

In [ ]:
output_file = "dimensionality_reduced.h5ad"
adata.write(output_file)

print("Processing completed.")
print(f"Saved output to: {output_file}")